In [ ]:
!pip install git+https://github.com/huggingface/transformers accelerate qwen-vl-utils torchvision


  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-kmgm3plo
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-kmgm3plo
  Resolved https://github.com/huggingface/transformers to commit 6abd9725ee7d809dc974991f8ff6c958afb63a3a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 21.1 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.5.0.dev0-py3-none-any.whl size=11251140 sha256=f657d582a1c267d36e1c6ba6eb1a1100d51a772237df4830e96ef3452153c1ef
  Stored in directory: /tmp/pip-ephem-wheel-cache-a57pz6lg/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import torch
import re
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# --- 1. Configuration ---

# >>> Updated to your new dataset path target <<<
folder_path = "/content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)"
output_file = "/content/drive/MyDrive/false_fire_structured_results.txt"

# Your exact requested prompt:
prompt_text = "Analyze the image for three specific data points: 1) Presence of fire/smoke. 2) Classification of the hazard. 3) Environmental setting. Synthesize these three points into exactly one concise descriptive phrase. Output nothing else."

# --- 2. Load the Qwen2-VL model and processor ---
print("Loading Model... This will take a moment.")
model_id = "Qwen/Qwen2-VL-2B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)
print(f"Model loaded successfully on {device}!")

# Helper function to extract "Meaningful Tokens"
def get_meaningful_tokens(text):
    words = re.findall(r'\b\w+\b', text.lower())
    stop_words = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'and', 'or', 'in', 'on', 'at', 'to', 'with', 'of', 'it', 'this', 'there'}
    return [w for w in words if w not in stop_words]

# --- 3. Analyze Images in the Folder ---
valid_extensions = {".jpg", ".jpeg", ".png"}

print(f"Looking for images in: {folder_path}")
if not os.path.exists(folder_path):
    print("ERROR: Folder not found! Check your Google Drive shortcut path.")
else:
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("=== STRUCTURED PROMPT RESULTS ===\n")
        f.write(f"Prompt Used: \"{prompt_text}\"\n")
        f.write(f"Dataset Path: {folder_path}\n\n")

        # Sort so it goes through your images alphabetically
        for filename in sorted(os.listdir(folder_path)):
            if not any(filename.lower().endswith(ext) for ext in valid_extensions):
                continue

            print(f"Processing: {filename} ...")
            image_path = os.path.join(folder_path, filename)

            # Prepare message for Qwen2-VL
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image_path},
                        {"type": "text", "text": prompt_text},
                    ],
                }
            ]

            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)

            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            )
            inputs = inputs.to(device)

            # Generate the response
            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=50)

            # Filter out the input tokens
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]

            phrase = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

            # Extract Metrics
            word_count = len(phrase.split())
            token_count = len(generated_ids_trimmed[0])
            meaningful_tokens = get_meaningful_tokens(phrase)

            # Write out to the text file
            f.write(f"File: {filename}\n")
            f.write(f"  Phrase: {phrase}\n")
            f.write(f"  Meaningful Tokens: {meaningful_tokens}\n")
            f.write(f"  Word Count: {word_count}\n")
            f.write(f"  Token Count: {token_count}\n\n")

    print(f"Finished processing! Results saved to Google Drive: {output_file}")


Loading Model... This will take a moment.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model loaded successfully on cuda!
Looking for images in: /content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)
Processing: 0087_FL_FWW_00100.jpg ...
Processing: 0087_FL_FWW_00150.jpg ...
Processing: 0088_FL_FWW_00001.jpg ...
Processing: 0089_FL_FWW_00001.jpg ...
Processing: 0121_FL_FWW_00126.jpg ...
Processing: 0132_FL_FWW_00089.jpg ...
Processing: 0150_FL_FWW_00256.jpg ...
Processing: 0229_FL_FWW_00269.jpg ...
Processing: 0328_FL_FWW_00293.jpg ...
Processing: 0338_FL_FWW_00328.jpg ...
Processing: AoF03409.jpg ...
Processing: PublicDataset00914.jpg ...
Processing: PublicDataset00924.jpg ...
Processing: PublicDataset01288.jpg ...
Processing: WEB00427.jpg ...
Processing: WEB02324.jpg ...
Processing: WEB03419.jpg ...
Processing: WEB07649 - Copy.jpg ...
Processing: WEB10106.jpg ...
Processing: WEB10129.jpg ...
Processing: cooking1.jpeg ...
Processing: cooking3.jpeg ...
Processing: cooking4.jpeg ...
Processing: cooking5.jpeg ...
Processing: ligther1.jpg ...
Processing: ligther2

OutOfMemoryError: CUDA out of memory. Tried to allocate 11.87 GiB. GPU 0 has a total capacity of 14.56 GiB of which 6.17 GiB is free. Including non-PyTorch memory, this process has 8.39 GiB memory in use. Of the allocated memory 6.12 GiB is allocated by PyTorch, and 2.13 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import os
import torch
import re
import gc # For memory garbage collection
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# --- 1. Configuration ---
folder_path = "/content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)"
output_file = "/content/drive/MyDrive/vision_model_results.txt"

prompt_text = "Analyze the image for three specific data points: 1) Presence of fire/smoke. 2) Classification of the hazard. 3) Environmental setting. Synthesize these three points into exactly one concise descriptive phrase. Output nothing else."

# --- 2. Load the Qwen2-VL model and processor ---
print("Loading Model... This will take a moment.")
model_id = "Qwen/Qwen2-VL-2B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)
print(f"Model loaded successfully on {device}!")

def get_meaningful_tokens(text):
    words = re.findall(r'\b\w+\b', text.lower())
    stop_words = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'and', 'or', 'in', 'on', 'at', 'to', 'with', 'of', 'it', 'this', 'there'}
    return [w for w in words if w not in stop_words]

# --- 3. Analyze Images in the Folder ---
valid_extensions = {".jpg", ".jpeg", ".png", ".webp", ".jfif", ".avif"}

print(f"Looking for images in: {folder_path}")
if not os.path.exists(folder_path):
    print("ERROR: Folder not found! Check your Google Drive shortcut path.")
else:
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("=== STRUCTURED PROMPT RESULTS ===\n")
        f.write(f"Prompt Used: \"{prompt_text}\"\n")
        f.write(f"Dataset Path: {folder_path}\n\n")

        for filename in sorted(os.listdir(folder_path)):
            if not any(filename.lower().endswith(ext) for ext in valid_extensions):
                continue

            print(f"Processing: {filename} ...")
            image_path = os.path.join(folder_path, filename)

            # Prepare message for Qwen2-VL
            # FIX 1: We added max_pixels to prevent massive images from blowing up VRAM
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image_path, "max_pixels": 501760},
                        {"type": "text", "text": prompt_text},
                    ],
                }
            ]

            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)

            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            )
            inputs = inputs.to(device)

            # Generate the response
            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=100)

            # Filter out the input tokens
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]

            phrase = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

            # Extract Metrics
            word_count = len(phrase.split())
            token_count = len(generated_ids_trimmed[0])
            meaningful_tokens = get_meaningful_tokens(phrase)

            # Write out to the text file
            f.write(f"File: {filename}\n")
            f.write(f"  Phrase: {phrase}\n")
            f.write(f"  Meaningful Tokens: {meaningful_tokens}\n")
            f.write(f"  Word Count: {word_count}\n")
            f.write(f"  Token Count: {token_count}\n\n")

            # FIX 2: Manually destroy the stored variables and flush the GPU memory cache!
            del inputs
            del generated_ids
            del generated_ids_trimmed
            del image_inputs
            gc.collect()
            torch.cuda.empty_cache()

    print(f"Finished processing! Results saved to Google Drive: {output_file}")


Loading Model... This will take a moment.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model loaded successfully on cuda!
Looking for images in: /content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)
Processing: 0087_FL_FWW_00100.jpg ...
Processing: 0087_FL_FWW_00150.jpg ...
Processing: 0088_FL_FWW_00001.jpg ...
Processing: 0089_FL_FWW_00001.jpg ...
Processing: 0121_FL_FWW_00126.jpg ...
Processing: 0132_FL_FWW_00089.jpg ...
Processing: 0150_FL_FWW_00256.jpg ...
Processing: 0229_FL_FWW_00269.jpg ...
Processing: 0328_FL_FWW_00293.jpg ...
Processing: 0338_FL_FWW_00328.jpg ...
Processing: AoF03409.jpg ...
Processing: PublicDataset00914.jpg ...
Processing: PublicDataset00924.jpg ...
Processing: PublicDataset01288.jpg ...
Processing: WEB00427.jpg ...
Processing: WEB02324.jpg ...
Processing: WEB03419.jpg ...
Processing: WEB07649 - Copy.jpg ...
Processing: WEB10106.jpg ...
Processing: WEB10129.jpg ...
Processing: cooking1.jpeg ...
Processing: cooking2.webp ...
Processing: cooking3.jpeg ...
Processing: cooking4.jpeg ...
Processing: cooking5.jpeg ...
Processing: ligther

# New Section

In [ ]:
import os
import torch
import re
import gc # For memory garbage collection
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# --- 1. Configuration ---
folder_path = "/content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)"
output_file = "/content/drive/MyDrive/vision_model_results.txt"

prompt_text = "Analyze the image for three specific data points: 1) Presence of fire/smoke. 2) Classification of the hazard. 3) Environmental setting. Synthesize these three points into exactly one concise descriptive phrase. Output nothing else."

# --- 2. Load the Qwen2-VL model and processor ---
print("Loading Model... This will take a moment.")
model_id = "Qwen/Qwen2-VL-2B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)
print(f"Model loaded successfully on {device}!")

def get_meaningful_tokens(text):
    words = re.findall(r'\b\w+\b', text.lower())
    stop_words = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'and', 'or', 'in', 'on', 'at', 'to', 'with', 'of', 'it', 'this', 'there'}
    return [w for w in words if w not in stop_words]

# --- 3. Analyze Images in the Folder ---
valid_extensions = {".jpg", ".jpeg", ".png", ".webp", ".jfif", ".avif"}

print(f"Looking for images in: {folder_path}")
if not os.path.exists(folder_path):
    print("ERROR: Folder not found! Check your Google Drive shortcut path.")
else:
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("=== STRUCTURED PROMPT RESULTS ===\n")
        f.write(f"Prompt Used: \"{prompt_text}\"\n")
        f.write(f"Dataset Path: {folder_path}\n\n")

        for filename in sorted(os.listdir(folder_path)):
            if not any(filename.lower().endswith(ext) for ext in valid_extensions):
                continue

            print(f"Processing: {filename} ...")
            image_path = os.path.join(folder_path, filename)

            # Prepare message for Qwen2-VL
            # FIX 1: We added max_pixels to prevent massive images from blowing up VRAM
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image_path, "max_pixels": 501760},
                        {"type": "text", "text": prompt_text},
                    ],
                }
            ]

            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)

            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            )
            inputs = inputs.to(device)

            # Generate the response
            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=100)

            # Filter out the input tokens
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]

            phrase = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

            # Extract Metrics
            word_count = len(phrase.split())
            token_count = len(generated_ids_trimmed[0])
            meaningful_tokens = get_meaningful_tokens(phrase)

            # Write out to the text file
            f.write(f"File: {filename}\n")
            f.write(f"  Phrase: {phrase}\n")
            f.write(f"  Meaningful Tokens: {meaningful_tokens}\n")
            f.write(f"  Word Count: {word_count}\n")
            f.write(f"  Token Count: {token_count}\n\n")

            # FIX 2: Manually destroy the stored variables and flush the GPU memory cache!
            del inputs
            del generated_ids
            del generated_ids_trimmed
            del image_inputs
            gc.collect()
            torch.cuda.empty_cache()

    print(f"Finished processing! Results saved to Google Drive: {output_file}")


In [ ]:
import os
import torch
import re
import gc # For memory garbage collection
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# --- 1. Configuration ---
folder_path = "/content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)"

# UPDATED ALIGNING TO THE PROMPT STYLE SO FILES DON'T OVERWRITE
output_file = "/content/drive/MyDrive/results_positive_prompt.txt"

# YOUR NEW EXACT REQUESTED PROMPT
prompt_text = "Describe only visible elements in the image in one short positive sentence. If fire is visible, describe its type and size."

# --- 2. Load the Qwen2-VL model and processor ---
print("Loading Model... This will take a moment.")
model_id = "Qwen/Qwen2-VL-2B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)
print(f"Model loaded successfully on {device}!")

def get_meaningful_tokens(text):
    words = re.findall(r'\b\w+\b', text.lower())
    stop_words = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'and', 'or', 'in', 'on', 'at', 'to', 'with', 'of', 'it', 'this', 'there'}
    return [w for w in words if w not in stop_words]

# --- 3. Analyze Images in the Folder ---
valid_extensions = {".jpg", ".jpeg", ".png", ".webp", ".jfif", ".avif"}

print(f"Looking for images in: {folder_path}")
if not os.path.exists(folder_path):
    print("ERROR: Folder not found! Check your Google Drive shortcut path.")
else:
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("=== SINGLE PROMPT RESULTS ===\n")
        f.write(f"Prompt Used: \"{prompt_text}\"\n")
        f.write(f"Dataset Path: {folder_path}\n\n")

        for filename in sorted(os.listdir(folder_path)):
            if not any(filename.lower().endswith(ext) for ext in valid_extensions):
                continue

            print(f"Processing: {filename} ...")
            image_path = os.path.join(folder_path, filename)

            # Prepare message for Qwen2-VL
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image_path, "max_pixels": 501760},
                        {"type": "text", "text": prompt_text},
                    ],
                }
            ]

            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)

            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            )
            inputs = inputs.to(device)

            # Generate the response
            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=100)

            # Filter out the input tokens
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]

            phrase = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

            # Extract Metrics
            word_count = len(phrase.split())
            token_count = len(generated_ids_trimmed[0])
            meaningful_tokens = get_meaningful_tokens(phrase)

            # Write out to the text file
            f.write(f"File: {filename}\n")
            f.write(f"  Phrase: {phrase}\n")
            f.write(f"  Meaningful Tokens: {meaningful_tokens}\n")
            f.write(f"  Word Count: {word_count}\n")
            f.write(f"  Token Count: {token_count}\n\n")

            # Manually destroy the stored variables and flush the GPU memory cache!
            del inputs
            del generated_ids
            del generated_ids_trimmed
            del image_inputs
            gc.collect()
            torch.cuda.empty_cache()

    print(f"Finished processing! Results saved to Google Drive: {output_file}")


Loading Model... This will take a moment.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model loaded successfully on cuda!
Looking for images in: /content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)
Processing: 0087_FL_FWW_00100.jpg ...
Processing: 0087_FL_FWW_00150.jpg ...
Processing: 0088_FL_FWW_00001.jpg ...
Processing: 0089_FL_FWW_00001.jpg ...
Processing: 0121_FL_FWW_00126.jpg ...
Processing: 0132_FL_FWW_00089.jpg ...
Processing: 0150_FL_FWW_00256.jpg ...
Processing: 0229_FL_FWW_00269.jpg ...
Processing: 0328_FL_FWW_00293.jpg ...
Processing: 0338_FL_FWW_00328.jpg ...
Processing: AoF03409.jpg ...
Processing: PublicDataset00914.jpg ...
Processing: PublicDataset00924.jpg ...
Processing: PublicDataset01288.jpg ...
Processing: WEB00427.jpg ...
Processing: WEB02324.jpg ...
Processing: WEB03419.jpg ...
Processing: WEB07649 - Copy.jpg ...
Processing: WEB10106.jpg ...
Processing: WEB10129.jpg ...
Processing: cooking1.jpeg ...
Processing: cooking2.webp ...
Processing: cooking3.jpeg ...
Processing: cooking4.jpeg ...
Processing: cooking5.jpeg ...
Processing: ligther

In [ ]:
import os
import torch
import re
import gc # For memory garbage collection
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# --- 1. Configuration ---
folder_path = "/content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)"

# UPDATED TO A NEW FILENAME
output_file = "/content/drive/MyDrive/results_original_prompt.txt"

# YOUR ORIGINAL REQUESTED PROMPT
prompt_text = "Describe this image in one short phrase focusing on whether fire is present, its nature (e.g. wildfire, campfire, flame, explosion, etc), and the surrounding environment. Reply with the phrase only."

# --- 2. Load the Qwen2-VL model and processor ---
print("Loading Model... This will take a moment.")
model_id = "Qwen/Qwen2-VL-2B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)
print(f"Model loaded successfully on {device}!")

def get_meaningful_tokens(text):
    words = re.findall(r'\b\w+\b', text.lower())
    stop_words = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'and', 'or', 'in', 'on', 'at', 'to', 'with', 'of', 'it', 'this', 'there'}
    return [w for w in words if w not in stop_words]

# --- 3. Analyze Images in the Folder ---
valid_extensions = {".jpg", ".jpeg", ".png", ".webp", ".jfif", ".avif"}

print(f"Looking for images in: {folder_path}")
if not os.path.exists(folder_path):
    print("ERROR: Folder not found! Check your Google Drive shortcut path.")
else:
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("=== SINGLE PROMPT RESULTS ===\n")
        f.write(f"Prompt Used: \"{prompt_text}\"\n")
        f.write(f"Dataset Path: {folder_path}\n\n")

        for filename in sorted(os.listdir(folder_path)):
            if not any(filename.lower().endswith(ext) for ext in valid_extensions):
                continue

            print(f"Processing: {filename} ...")
            image_path = os.path.join(folder_path, filename)

            # Prepare message for Qwen2-VL
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image_path, "max_pixels": 501760},
                        {"type": "text", "text": prompt_text},
                    ],
                }
            ]

            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)

            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            )
            inputs = inputs.to(device)

            # Generate the response
            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=100)

            # Filter out the input tokens
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]

            phrase = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

            # Extract Metrics
            word_count = len(phrase.split())
            token_count = len(generated_ids_trimmed[0])
            meaningful_tokens = get_meaningful_tokens(phrase)

            # Write out to the text file
            f.write(f"File: {filename}\n")
            f.write(f"  Phrase: {phrase}\n")
            f.write(f"  Meaningful Tokens: {meaningful_tokens}\n")
            f.write(f"  Word Count: {word_count}\n")
            f.write(f"  Token Count: {token_count}\n\n")

            # Manually destroy the stored variables and flush the GPU memory cache!
            del inputs
            del generated_ids
            del generated_ids_trimmed
            del image_inputs
            gc.collect()
            torch.cuda.empty_cache()

    print(f"Finished processing! Results saved to Google Drive: {output_file}")


Loading Model... This will take a moment.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model loaded successfully on cuda!
Looking for images in: /content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)
Processing: 0087_FL_FWW_00100.jpg ...
Processing: 0087_FL_FWW_00150.jpg ...
Processing: 0088_FL_FWW_00001.jpg ...
Processing: 0089_FL_FWW_00001.jpg ...
Processing: 0121_FL_FWW_00126.jpg ...
Processing: 0132_FL_FWW_00089.jpg ...
Processing: 0150_FL_FWW_00256.jpg ...
Processing: 0229_FL_FWW_00269.jpg ...
Processing: 0328_FL_FWW_00293.jpg ...
Processing: 0338_FL_FWW_00328.jpg ...
Processing: AoF03409.jpg ...
Processing: PublicDataset00914.jpg ...
Processing: PublicDataset00924.jpg ...
Processing: PublicDataset01288.jpg ...
Processing: WEB00427.jpg ...
Processing: WEB02324.jpg ...
Processing: WEB03419.jpg ...
Processing: WEB07649 - Copy.jpg ...
Processing: WEB10106.jpg ...
Processing: WEB10129.jpg ...
Processing: cooking1.jpeg ...
Processing: cooking2.webp ...
Processing: cooking3.jpeg ...
Processing: cooking4.jpeg ...
Processing: cooking5.jpeg ...
Processing: ligther

In [ ]:
import os
import torch
import re
import gc # For memory garbage collection
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# --- 1. Configuration ---
folder_path = "/content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)"

# UPDATED FILENAME FOR THIS TEST
output_file = "/content/drive/MyDrive/results_keyword_prompt.txt"

# YOUR NEW EXACT REQUESTED PROMPT
prompt_text = "Describe the primary visible objects or events in this scene using a few short keywords. Do not describe things that are not present."

# --- 2. Load the Qwen2-VL model and processor ---
print("Loading Model... This will take a moment.")
model_id = "Qwen/Qwen2-VL-2B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)
print(f"Model loaded successfully on {device}!")

def get_meaningful_tokens(text):
    words = re.findall(r'\b\w+\b', text.lower())
    stop_words = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'and', 'or', 'in', 'on', 'at', 'to', 'with', 'of', 'it', 'this', 'there'}
    return [w for w in words if w not in stop_words]

# --- 3. Analyze Images in the Folder ---
valid_extensions = {".jpg", ".jpeg", ".png", ".webp", ".jfif", ".avif"}

print(f"Looking for images in: {folder_path}")
if not os.path.exists(folder_path):
    print("ERROR: Folder not found! Check your Google Drive shortcut path.")
else:
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("=== SINGLE PROMPT RESULTS ===\n")
        f.write(f"Prompt Used: \"{prompt_text}\"\n")
        f.write(f"Dataset Path: {folder_path}\n\n")

        for filename in sorted(os.listdir(folder_path)):
            if not any(filename.lower().endswith(ext) for ext in valid_extensions):
                continue

            print(f"Processing: {filename} ...")
            image_path = os.path.join(folder_path, filename)

            # Prepare message for Qwen2-VL
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image_path, "max_pixels": 501760},
                        {"type": "text", "text": prompt_text},
                    ],
                }
            ]

            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)

            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            )
            inputs = inputs.to(device)

            # Generate the response
            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=100)

            # Filter out the input tokens
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]

            phrase = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

            # Extract Metrics
            word_count = len(phrase.split())
            token_count = len(generated_ids_trimmed[0])
            meaningful_tokens = get_meaningful_tokens(phrase)

            # Write out to the text file
            f.write(f"File: {filename}\n")
            f.write(f"  Phrase: {phrase}\n")
            f.write(f"  Meaningful Tokens: {meaningful_tokens}\n")
            f.write(f"  Word Count: {word_count}\n")
            f.write(f"  Token Count: {token_count}\n\n")

            # Manually destroy the stored variables and flush the GPU memory cache!
            del inputs
            del generated_ids
            del generated_ids_trimmed
            del image_inputs
            gc.collect()
            torch.cuda.empty_cache()

    print(f"Finished processing! Results saved to Google Drive: {output_file}")


Loading Model... This will take a moment.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model loaded successfully on cuda!
Looking for images in: /content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)
Processing: 0087_FL_FWW_00100.jpg ...
Processing: 0087_FL_FWW_00150.jpg ...
Processing: 0088_FL_FWW_00001.jpg ...
Processing: 0089_FL_FWW_00001.jpg ...
Processing: 0121_FL_FWW_00126.jpg ...
Processing: 0132_FL_FWW_00089.jpg ...
Processing: 0150_FL_FWW_00256.jpg ...
Processing: 0229_FL_FWW_00269.jpg ...
Processing: 0328_FL_FWW_00293.jpg ...
Processing: 0338_FL_FWW_00328.jpg ...
Processing: AoF03409.jpg ...
Processing: PublicDataset00914.jpg ...
Processing: PublicDataset00924.jpg ...
Processing: PublicDataset01288.jpg ...
Processing: WEB00427.jpg ...
Processing: WEB02324.jpg ...
Processing: WEB03419.jpg ...
Processing: WEB07649 - Copy.jpg ...
Processing: WEB10106.jpg ...
Processing: WEB10129.jpg ...
Processing: cooking1.jpeg ...
Processing: cooking2.webp ...
Processing: cooking3.jpeg ...
Processing: cooking4.jpeg ...
Processing: cooking5.jpeg ...
Processing: ligther

In [3]:
import os
import time  # NEW: Added for tracking processing speed
import torch
import re
import gc # For memory garbage collection
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# --- 1. Configuration ---
folder_path = "/content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)"

# UPDATED FILENAME TO SAVE THE TIMING DATA
output_file = "/content/drive/MyDrive/results_with_timing.txt"

# A clean, specialized prompt test
prompt_text = "Provide a single, concise phrase describing the presence or absence of fire and smoke. If present, specify its type (e.g., wildfire, car fire, explosion) and the surrounding scene. Do not include introductory text."

# --- 2. Load the Qwen2-VL model and processor ---
print("Loading Model... This will take a moment.")
model_id = "Qwen/Qwen2-VL-2B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)
print(f"Model loaded successfully on {device}!")

def get_meaningful_tokens(text):
    words = re.findall(r'\b\w+\b', text.lower())
    stop_words = {'the', 'a', 'an', 'is', 'are', 'was', 'were', 'and', 'or', 'in', 'on', 'at', 'to', 'with', 'of', 'it', 'this', 'there'}
    return [w for w in words if w not in stop_words]

# --- 3. Analyze Images in the Folder ---
valid_extensions = {".jpg", ".jpeg", ".png", ".webp", ".jfif", ".avif"}

print(f"Looking for images in: {folder_path}")
if not os.path.exists(folder_path):
    print("ERROR: Folder not found! Check your Google Drive shortcut path.")
else:
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("=== PROMPT RESULTS (WITH TIMING) ===\n")
        f.write(f"Prompt Used: \"{prompt_text}\"\n")
        f.write(f"Dataset Path: {folder_path}\n\n")

        for filename in sorted(os.listdir(folder_path)):
            if not any(filename.lower().endswith(ext) for ext in valid_extensions):
                continue

            image_path = os.path.join(folder_path, filename)

            # Prepare message for Qwen2-VL
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "image", "image": image_path, "max_pixels": 501760},
                        {"type": "text", "text": prompt_text},
                    ],
                }
            ]

            text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
            image_inputs, video_inputs = process_vision_info(messages)

            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            )
            inputs = inputs.to(device)

            # -----------------------------------------------
            # --- START THE STOPWATCH FOR GENERATION ---
            start_time = time.time()

            # Generate the response
            with torch.no_grad():
                generated_ids = model.generate(**inputs, max_new_tokens=100)

            # --- STOP THE STOPWATCH! ---
            end_time = time.time()
            processing_time = end_time - start_time
            # -----------------------------------------------

            # Filter out the input tokens
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]

            phrase = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

            # Extract Metrics
            word_count = len(phrase.split())
            token_count = len(generated_ids_trimmed[0])
            meaningful_tokens = get_meaningful_tokens(phrase)

            # We now print exactly how long it took directly to your screen!
            print(f"Processed: {filename} in {processing_time:.2f} seconds")

            # Write out to the text file (including the new timing stat)
            f.write(f"File: {filename}\n")
            f.write(f"  Phrase: {phrase}\n")
            f.write(f"  Meaningful Tokens: {meaningful_tokens}\n")
            f.write(f"  Word Count: {word_count}\n")
            f.write(f"  Token Count: {token_count}\n")
            f.write(f"  Processing Time: {processing_time:.2f} seconds\n\n")

            # Manually destroy the stored variables and flush the GPU memory cache!
            del inputs
            del generated_ids
            del generated_ids_trimmed
            del image_inputs
            gc.collect()
            torch.cuda.empty_cache()

    print(f"Finished processing! Results saved to Google Drive: {output_file}")


Loading Model... This will take a moment.


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Model loaded successfully on cuda!
Looking for images in: /content/drive/MyDrive/fire_detection/PickTheModelDataSet(30pics)
Processed: 0087_FL_FWW_00100.jpg in 1.06 seconds
Processed: 0087_FL_FWW_00150.jpg in 1.05 seconds
Processed: 0088_FL_FWW_00001.jpg in 0.99 seconds
Processed: 0089_FL_FWW_00001.jpg in 1.19 seconds
Processed: 0121_FL_FWW_00126.jpg in 1.10 seconds
Processed: 0132_FL_FWW_00089.jpg in 1.28 seconds
Processed: 0150_FL_FWW_00256.jpg in 0.78 seconds
Processed: 0229_FL_FWW_00269.jpg in 1.36 seconds
Processed: 0328_FL_FWW_00293.jpg in 1.20 seconds
Processed: 0338_FL_FWW_00328.jpg in 1.33 seconds
Processed: AoF03409.jpg in 0.95 seconds
Processed: PublicDataset00914.jpg in 0.70 seconds
Processed: PublicDataset00924.jpg in 0.98 seconds
Processed: PublicDataset01288.jpg in 0.71 seconds
Processed: WEB00427.jpg in 0.62 seconds
Processed: WEB02324.jpg in 1.09 seconds
Processed: WEB03419.jpg in 1.21 seconds
Processed: WEB07649 - Copy.jpg in 1.15 seconds
Processed: WEB10106.jpg in 1.